# Capstone Project: California House Price Prediction

**Project type:** regression  
**Goal:** predict a California district's median house value, measured in $100,000s.

This notebook uses a reproducible train/validation/test workflow. The test set stays untouched until the final model has been selected.

## 1. Imports and reproducibility

The notebook does not import or call SciPy directly. Scikit-learn handles the estimators and NumPy handles the reported metrics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.datasets import fetch_california_housing
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
print(f"Random seed: {RANDOM_SEED}")

## 2. Load and validate the data source

`fetch_california_housing` downloads the archive from scikit-learn's registered Figshare URL on first use and then caches it locally. Scikit-learn verifies the registered archive checksum during download. See the [dataset documentation](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset). A first run therefore needs network access; later runs can use the cache.

We do not silently accept an unexpected download or schema: the next cell checks row count, exact columns, missing values, numeric finiteness, and target bounds before modeling.

In [ ]:
try:
    housing = fetch_california_housing(as_frame=True)
except OSError as exc:
    raise RuntimeError(
        "California Housing is not cached. Connect to the network for the first run, "
        "then rerun this cell so scikit-learn can download and verify the dataset."
    ) from exc

df = housing.frame.copy()
expected_features = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms",
    "Population", "AveOccup", "Latitude", "Longitude",
]
expected_columns = expected_features + ["MedHouseVal"]

assert df.shape == (20_640, 9), f"Unexpected shape: {df.shape}"
assert df.columns.tolist() == expected_columns, f"Unexpected columns: {df.columns.tolist()}"
assert not df.isna().any().any(), "Unexpected missing values"
assert np.isfinite(df.to_numpy()).all(), "Unexpected non-finite values"
assert df["MedHouseVal"].between(0, 5.00001).all(), "Target is outside documented bounds"

print("Validated dataset shape:", df.shape)
df.head()

## 3. Split before exploration

The 20% test set is locked away first. A second split creates a validation set for model selection. All plots, preprocessing fits, and model comparisons below use only training or validation rows; the test rows are evaluated once at the end.

In [ ]:
X = df[expected_features]
y = df["MedHouseVal"]

X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development, y_development, test_size=0.25, random_state=RANDOM_SEED
)

assert set(X_train.index).isdisjoint(X_validation.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_validation.index).isdisjoint(X_test.index)
print("Train / validation / test rows:", len(X_train), len(X_validation), len(X_test))

## 4. Training-set exploration

The capped target at 5 means `$500,000 or more`, a limitation to remember when interpreting predictions.

In [ ]:
train_df = X_train.assign(MedHouseVal=y_train)
display(train_df.describe().T.round(3))

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(train_df["MedHouseVal"], bins=40, color="#2a6f97", ax=ax)
ax.set(
    title="Training-set distribution of median house value",
    xlabel="Median house value ($100,000s)",
    ylabel="District count",
)
plt.tight_layout()
plt.show()

In [ ]:
plot_sample = train_df.sample(n=3_000, random_state=RANDOM_SEED)
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=plot_sample, x="MedInc", y="MedHouseVal",
    alpha=0.35, s=24, color="#ca6702", edgecolor=None, ax=ax,
)
ax.set(
    title="Median income and house value (fixed training sample)",
    xlabel="Median income (tens of thousands of dollars)",
    ylabel="Median house value ($100,000s)",
)
plt.tight_layout()
plt.show()

In [ ]:
correlations = train_df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(
    correlations, annot=True, fmt=".2f", cmap="vlag", center=0,
    square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax,
)
ax.set_title("Training-set Pearson correlations", pad=14)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Baseline and candidate models

The median predictor is a necessary reality check. The linear model uses a `Pipeline`, so `StandardScaler` learns means and standard deviations from the training split only. The tree does not need scaling. Both candidates and their hyperparameters are declared before validation scoring.

In [ ]:
def regression_metrics(actual, predicted):
    actual_array = np.asarray(actual, dtype=float)
    predicted_array = np.asarray(predicted, dtype=float)
    residual = actual_array - predicted_array
    return {
        "RMSE": float(np.sqrt(np.mean(residual ** 2))),
        "MAE": float(np.mean(np.abs(residual))),
        "R2": float(1 - np.sum(residual ** 2) / np.sum((actual_array - actual_array.mean()) ** 2)),
    }

models = {
    "Median baseline": DummyRegressor(strategy="median"),
    "Linear regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LinearRegression()),
    ]),
    "Decision tree (depth 8)": DecisionTreeRegressor(
        max_depth=8, min_samples_leaf=10, random_state=RANDOM_SEED
    ),
}

validation_rows = []
fitted_models = {}
for name, model in models.items():
    fitted = clone(model).fit(X_train, y_train)
    fitted_models[name] = fitted
    validation_rows.append({
        "Model": name,
        **regression_metrics(y_validation, fitted.predict(X_validation)),
    })

validation_results = (
    pd.DataFrame(validation_rows).set_index("Model").sort_values("RMSE")
)
validation_results.round(3)

## 6. Select, refit, and evaluate once on the test set

Selection uses validation RMSE only. The selected pipeline is then freshly refit on the combined training and validation rows. No statistic is fit on the test features or target.

In [ ]:
selected_name = validation_results.index[0]
selected_model = clone(models[selected_name]).fit(X_development, y_development)
test_predictions = selected_model.predict(X_test)
test_metrics = pd.Series(regression_metrics(y_test, test_predictions), name="Test score")

print("Selected model:", selected_name)
display(test_metrics.to_frame().round(3))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 5))
limits = [min(y_test.min(), test_predictions.min()), max(y_test.max(), test_predictions.max())]
axes[0].scatter(y_test, test_predictions, alpha=0.25, s=18, color="#2a6f97")
axes[0].plot(limits, limits, "--", color="#bb3e03", linewidth=2, label="Perfect prediction")
axes[0].set(
    title="Held-out test predictions",
    xlabel="Actual value ($100,000s)",
    ylabel="Predicted value ($100,000s)",
)
axes[0].legend()

residuals = np.asarray(y_test) - test_predictions
sns.histplot(residuals, bins=40, color="#ca6702", ax=axes[1])
axes[1].axvline(0, linestyle="--", color="#333333", linewidth=1.5)
axes[1].set(
    title="Held-out test residuals",
    xlabel="Actual minus predicted ($100,000s)",
    ylabel="District count",
)
figure.suptitle(selected_name, fontsize=14, y=1.02)
figure.tight_layout()
plt.show()

## 7. Conclusions

The following summary is generated from the saved run rather than typed from an earlier experiment. Remember that the target is capped, so the model cannot learn variation above $500,000 from this dataset.

In [ ]:
strongest_feature = (
    correlations["MedHouseVal"].drop("MedHouseVal").abs().idxmax()
)
print(f"1. Best validation model: {selected_name}.")
print(f"2. Test RMSE: {test_metrics['RMSE']:.3f} ($100,000s).")
print(f"3. Strongest absolute training-set correlation with the target: {strongest_feature}.")
print(f"4. Mean test residual: {residuals.mean():.3f}; inspect the plots for non-random structure.")
print("5. Next: cross-validated tree ensembles and geographic feature engineering, while preserving the held-out test set.")